This code builds on the transient heat conduction example from the Pyomo tutorial examples by Jeff Kantor. The model simulates the following equation:

$\frac{\partial T}{\partial t} = \alpha \frac{ \partial ^2 T}{\partial x^2}$

$\alpha = 0.5$

$T(0,x) = 0$ for all $0 \leq x \leq 1$

$T(t,1) = 1$ for all $t >0$

$\frac{dT}{dx}(t,0) = u$ for all $t \geq 0$

The example was solved to run a doe on the diffusion process. u is the control input and needs to be computed as the output of the run_doe. The optimal value of u should be 0 for all t

In [1]:
# ## Importing necessary files and packages

# %matplotlib inline 

# ## show all matplotlib plots directly inside the notebook cell output other options instead of widget,notebook

# import numpy as np
# import matplotlib.pyplot as plt
# from mpl_toolkits.mplot3d.axes3d import Axes3D ## Needed to create 3D plots

# import shutil ## Shell utilities for python, allows to copy/move/rename files or delete directories
# import sys ## interact with python interpreter itself
# import os.path ## path manipulation utilities

# if not shutil.which("pyomo"):
#     !pip install -q pyomo
#     assert(shutil.which("pyomo")) ## Check if pyomo is installed, if not, install it and double check installation

# from pyomo.environ import *
# from pyomo.dae import *
    

In [2]:
# ## First defining a function that creates a wireframe plot from a pyomo.dae model


# def model_plot(m):
#     x = sorted(m.x) # sort the x axis so that the set is monotonic- ease of plotting
#     t = sorted(m.t) # sort the t axis

#     ## Define grid variables

#     xgrid = np.zeros((len(t),len(x)))
#     tgrid = np.zeros((len(t),len(x)))
#     Tgrid = np.zeros((len(t),len(x)))

#     ## assigning values to the grid variables

#     for index_t in range(0,len(t)):
#         for index_x in range(0,len(x)):
#             xgrid[index_t,index_x] = x[index_x]
           
#             tgrid[index_t,index_x] = t[index_t]
         
#             Tgrid[index_t,index_x] = m.T[t[index_t],x[index_x]].value
            

#     fig = plt.figure(figsize =(10,6))
#     ax = fig.add_subplot(1,1,1, projection = '3d')
#     ax.set_xlabel('Distance')
#     ax.set_ylabel('Time')
#     ax.set_zlabel('Temperature')
    # p = ax.plot_wireframe(xgrid,tgrid,Tgrid)

In [3]:
# ## Creating and solving the model


# m = ConcreteModel() ## instantiating model

# m.x = ContinuousSet(bounds = (0,1)) # instantiating x set
# m.t = ContinuousSet(bounds = (0,2)) # instantiating t set

# m.T = Var(m.t,m.x)

# m.dTdt = DerivativeVar(m.T, wrt = m.t)
# m.dTdx = DerivativeVar(m.T, wrt = m.x)
# m.d2Tdx2 = DerivativeVar(m.T, wrt = (m.x,m.x)) #DerivativeVar(m.dTdx, wrt = m.x) ## Alternatively,  

# @m.Constraint(m.t,m.x) ## create a 2-dimensional constraint block where indices are m.t and m.x

# def pde(m,t,x):
#     if t == 0:
#         return Constraint.Skip ## initial conditions
#     if x == 0 or x == 1:
#         return Constraint.Skip ## boundary conditions
#     return m.dTdt[t,x] == 0.5*m.d2Tdx2[t,x]


# ## Define objective function

# m.obj = Objective(expr = 1) # initialize an objective value of 1

# m.ic = Constraint(m.x,rule = lambda m, x: m.T[0,x] == 0 if x> 0 and x<1 else Constraint.Skip) ## Defining initial conditions

# ## Defining the boundary conditions

# m.bc1 = Constraint(m.t, rule = lambda m, t: m.T[t,1] == 1)
# m.bc2 = Constraint(m.t, rule = lambda m, t: m.dTdx[t,0] == 0)

# ## Discretizing the PDE in space and time
# TransformationFactory('dae.finite_difference').apply_to(m,nfe = 50, scheme = 'FORWARD', wrt = m.x)
# TransformationFactory('dae.finite_difference').apply_to(m,nfe = 50, scheme = 'FORWARD', wrt = m.t)


# # SolverFactory('ipopt').solve(m,tee = True).write()

# SolverFactory('ipopt', options={'linear_solver': 'mumps'}).solve(m, tee=False).write()

# # for idx in m.T:
# #     print(idx, value(m.T[idx]))

# # model_plot(m)


        

In [4]:
## Now running the Pyomo Model

import pyomo.environ as pyo
from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
import numpy as np
from pyomo.contrib.parmest.experiment import Experiment



# ========================
class PDE_model(Experiment):
    def __init__(self, data, nfe, ncp):
        """
        Arguments
        ---------
        data: object containing vital experimental information
        nfe: number of finite elements
        ncp: number of collocation points for the finite elements
        """
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None

        #############################
        # End constructor definition

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    # Create flexible model without data
    def create_model(self):
        """
        This is an example user model provided to DoE library.
        It is a dynamic problem solved by Pyomo.DAE.

        Return
        ------
        m: a Pyomo.DAE model
        """

        m = self.model = pyo.ConcreteModel()

        # Model parameters
     

        # Define model variables
        ########################
        # time
        m.t = ContinuousSet(bounds=[0, 1])
        m.x = ContinuousSet(bounds=[0, 1])

        # input
        m.u = pyo.Var(m.t, within=pyo.Reals,initialize = 0.1, bounds=(0, 50))
        m.T = pyo.Var(m.t, m.x, within=pyo.Reals, bounds=(0, 1))
        # Unknown parameter
        m.alpha = pyo.Var(within=pyo.Reals)

        # Differential variables 
        m.dTdt = DerivativeVar(m.T, wrt=m.t)
        m.dTdx = DerivativeVar(m.T, wrt=m.x)
        m.d2Tdx2 = DerivativeVar(m.T, wrt=(m.x,m.x))

        ########################
        # End variable def.

        # Equation definition
        ########################

        # Expression for the state evolution


        @m.Constraint(m.t,m.x)

        
        def pde(m,t,x):
            if t == 0:
                return Constraint.Skip ## initial conditions
            if x == 0 or x == 1:
                return Constraint.Skip ## boundary conditions
            return m.dTdt[t,x] == 1*m.d2Tdx2[t,x]


       


        ########################
        # End equation definition

    def finalize_model(self):
        """
        """
        m = self.model

        m.alpha.fix(self.data["alpha"])

        # Initial condition

        m.ic = Constraint(m.x,rule = lambda m, x: m.T[0,x] == 0 if x> 0 and x<1 else Constraint.Skip) ## Defining initial conditions

        ## Defining the boundary conditions

        m.bc1 = Constraint(m.t, rule = lambda m, t: m.T[t,1] == 1)
        m.bc2 = Constraint(m.t, rule = lambda m, t: m.dTdx[t,0] == m.u[t])


        # Update model time `t` with time range and control time points
        # m.t.update(self.data["t_range"])
        # m.t.update(list(m.t))
        # multiple = self.data["u_multiple"]
        # for idx in m.u:
        #     m.u[idx].set_value(multiple)

        from pyomo.environ import value

        # print("Uninitialized u entries:")
        for idx in m.u:
            if m.u[idx].value is None:
                print("  u[{}] is None".format(idx))

        # Discretizing the model
        # discr = pyo.TransformationFactory("dae.collocation")
        discr = pyo.TransformationFactory("dae.finite_difference")
        discr.apply_to(m, nfe=self.nfe, scheme = 'CENTRAL',  wrt=m.t)
        discr.apply_to(m, nfe=self.nfe, scheme = 'CENTRAL', wrt=m.x)


        #########################
        # End model finalization

    def label_experiment(self):
        """
        Example for annotating (labeling) the model with a
        full experiment.
        """
        m = self.model

        # Set measurement labels
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        
        m.experiment_outputs.update((m.T[t,x], None) for t in m.t for x in m.x)


        # Adding error for measurement values (assuming no covariance and constant error for all measurements)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_error = 1e-2  # Error in state measurement
      
        m.measurement_error.update((m.T[t,x], meas_error) for t in m.t for x in m.x)
   

        # Identify design variables (experiment inputs) for the model
        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add experimental input label for control input
        m.experiment_inputs.update((m.u[t], None) for t in m.t)

        # Add unknown parameter labels
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add labels to all unknown parameters with nominal value as the value
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.alpha])

        #########################
        # End model labeling

In [5]:
#  ___________________________________________________________________________
#
#  Pyomo: Python Optimization Modeling Objects
#  Copyright (c) 2008-2025
#  National Technology and Engineering Solutions of Sandia, LLC
#  Under the terms of Contract DE-NA0003525 with National Technology and
#  Engineering Solutions of Sandia, LLC, the U.S. Government retains certain
#  rights in this software.
#  This software is distributed under the 3-clause BSD License.
#  ___________________________________________________________________________
from pyomo.common.dependencies import numpy as np, pathlib

from pyomo.contrib.doe.examples.reactor_experiment import ReactorExperiment
from pyomo.contrib.doe import DesignOfExperiments

import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    TransformationFactory,
    SolverFactory,
    Objective,
    minimize,
    value as pyovalue,
    Suffix,
    Expression,
    sin,
    NonNegativeReals,
)



data_ex = {"u_multiple": 0.5, "alpha": 0.9}

# Create an Experiment object; data and discretization information are part
# of the constructor of this object
experiment = PDE_model(data=data_ex, nfe=10, ncp=0)
from pyomo.environ import value

        # print("Uninitialized u entries:")


# Use a central difference, with step size 1e-3
fd_formula = "central"
step_size = 1e-3

# Use the determinant objective with scaled sensitivity matrix
objective_option = "determinant"
scale_nominal_param_value = True

# Create the DesignOfExperiments object
# We will not be passing any prior information in this example.
# We also will rely on the initialization routine within
# the DesignOfExperiments class.
doe_obj = DesignOfExperiments(
    experiment,
    fd_formula=fd_formula,
    step=step_size,
    objective_option=objective_option,
    gradient_method = "pynumero",
    scale_constant_value=1,
    scale_nominal_param_value=scale_nominal_param_value,
    prior_FIM=None,
    jac_initial=None,
    fim_initial=None,
    L_diagonal_lower_bound=1e-7,
    solver=SolverFactory('IPOPT', options={'linear_solver': 'mumps'}),
    tee=True,
    get_labeled_model_args=None,
    _Cholesky_option=True,
    _only_compute_fim_lower=True,
)

# # Make design ranges to compute the full factorial design
# design_ranges = {"CA[0]": [1, 5, 9], "T[0]": [300, 700, 9]}

# # Compute the full factorial design with the sequential FIM calculation
# doe_obj.compute_FIM_full_factorial(design_ranges=design_ranges, method="sequential")

# # Plot the results
# doe_obj.draw_factorial_figure(
#     sensitivity_design_variables=["CA[0]", "T[0]"],
#     fixed_design_variables={
#         "T[0.125]": 300,
#         "T[0.25]": 300,
#         "T[0.375]": 300,
#         "T[0.5]": 300,
#         "T[0.625]": 300,
#         "T[0.75]": 300,
#         "T[0.875]": 300,
#         "T[1]": 300,
#     },
#     title_text="Reactor Example",
#     xlabel_text="Concentration of A (M)",
#     ylabel_text="Initial Temperature (K)",
#     figure_file_name="example_reactor_compute_FIM",
#     log_scale=False,
# )


doe_obj.run_doe()

print(doe_obj.results["Experiment Design"])

# Print out a results summary
# print("Optimal experiment values: ")
# print(
#     "\tInitial concentration: {:.2f}".format(
#         doe_obj.results["Experiment Design"]
#     )
# )
print(
    ("\tControl Inputs: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
        *doe_obj.results["Experiment Design"]
    )
)
print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
print(
    "Objective value at optimal design: {:.2f}".format(
        pyo.value(doe_obj.model.objective)
    )
)



print(sorted(doe_obj.results.keys()))

print("Solve time (s):", doe_obj.results["Solve Time"])
print("Build time (s):", doe_obj.results["Build Time"])
print("Initialization time (s):", doe_obj.results["Initialization Time"])
print("Total wall time (s):", doe_obj.results["Wall-clock Time"])


TypeError: DesignOfExperiments.__init__() got an unexpected keyword argument 'gradient_method'